In [ ]:
!pip -q install transformers pillow 
from transformers import BlipProcessor, BlipForConditionalGeneration 

import os
from PIL import Image
from google.colab import files 

# 1. Defina o caminho correto
pasta_imagens = "/content/atelie-generativo-pequi-do-cerrado/dados/"
caminho_arquivo_texto = "/content/atelie-generativo-pequi-do-cerrado/dados/legendas.txt"
token = "estilo_rupestre, "

# --- MODO DETETIVE ATIVADO ---
print(f"Olhando dentro da pasta: {pasta_imagens}")
try:
    arquivos_encontrados = os.listdir(pasta_imagens)
    print(f"Achei {len(arquivos_encontrados)} arquivos aqui dentro.")
    print(f"São eles: {arquivos_encontrados}\n")
except FileNotFoundError:
    print("ERRO: A pasta 'dados' não foi encontrada. Verifique se ela existe com esse exato nome no seu repositório do GitHub.")
    arquivos_encontrados = []
# -----------------------------

print("-" * 40)
print("Iniciando a leitura das imagens...\n")

with open(caminho_arquivo_texto, "w") as arquivo_legendas:
    for nome_do_arquivo in arquivos_encontrados:
        
        # O .lower() força o nome para minúsculo, assim pegamos .JPG, .PNG, etc.
        if nome_do_arquivo.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            caminho_completo = os.path.join(pasta_imagens, nome_do_arquivo)
            print(f"Lendo e traduzindo: {nome_do_arquivo}")
            
            img = Image.open(caminho_completo).convert("RGB")
            saida = blip.generate(**proc(img, return_tensors="pt").to("cuda"), max_new_tokens=40)
            texto_da_ia = proc.decode(saida[0], skip_special_tokens=True)
            
            legenda_final = f"{nome_do_arquivo}: {token}{texto_da_ia}\n"
            arquivo_legendas.write(legenda_final)

print("\nGeração concluída! Baixando o arquivo para o seu computador...")

# Se o arquivo tiver algum texto, ele faz o download
if os.path.exists(caminho_arquivo_texto) and os.path.getsize(caminho_arquivo_texto) > 0:
    files.download(caminho_arquivo_texto)
else:
    print("Nenhuma imagem compatível foi encontrada para gerar o arquivo.")